In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import dayplot
import pandas as pd

import os
import sys

# Add the parent directory to sys.path so we can import server_config
notebook_dir = os.getcwd()  # current working directory
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
sys.path.append(parent_dir)

# Now import server_config directly (without the dot)
from server_config import datapath


In [ ]:
df = pd.read_feather(datapath + '/preprocessed/backup_passive_recent.feather')

In [ ]:
df

In [ ]:
def plot_calendar_by_year(df_dayplot, date_col="startTimestamp_day", value_col="n_samples_per_day", 
                          cmap="inferno", title=None, figsize_per_year=3, plot_legend=False, cal_kwargs={}):
    """
    Create calendar plots split by year with year labels.
    
    Parameters:
    -----------
    df_dayplot : pd.DataFrame
        DataFrame with dates and values to plot
    date_col : str
        Name of the column containing dates
    value_col : str
        Name of the column containing values to plot
    cmap : str
        Colormap name (e.g., 'Reds', 'Greens', 'inferno')
    title : str, optional
        Title for the overall figure
    figsize_per_year : int
        Height in inches per year subplot
    """
    # Get the unique years in the data
    years = sorted(df_dayplot[date_col].dt.year.unique())
    
    # Create subplots - one for each year
    fig, axes = plt.subplots(nrows=len(years), figsize=(16, figsize_per_year * len(years)))
    
    # Handle case where there's only one year
    if len(years) == 1:
        axes = [axes]
    
    # Create a calendar plot for each year
    for i, year in enumerate(years):
        if plot_legend:
            legend = (i == len(years) - 1) # Show legend only for the last subplot
        else:
            legend = False
        dayplot.calendar(
            dates=df_dayplot[date_col],
            values=df_dayplot[value_col],
            start_date=f"{year}-01-01",
            end_date=f"{year}-12-31",
            cmap=cmap,
            ax=axes[i],
            week_starts_on="Monday",
            legend=legend,
            legend_bins=8,
            legend_labels="auto",
            **cal_kwargs
        )
        
        # Add year label on the left
        text_args = dict(x=-4, y=3.5, size=30, rotation=90, color="#aaa", va="center")
        axes[i].text(s=str(year), **text_args)
    
    if title:
        fig.suptitle(title, fontsize=20, y=1.02)
    
    plt.tight_layout()
    # plt.show()
    return fig, axes


In [ ]:
type2plot = "HeartRate"
df_filtered = df[df["type"] == type2plot]
n_subjects = df_filtered["customer"].nunique()
df_dayplot = df_filtered.groupby("startTimestamp_day").size().reset_index(name="n_samples_per_day")

In [ ]:
# n_subjects = df_dayplot["cu"]

plot_calendar_by_year(
    df_dayplot,
    cmap="Reds",
    title=f"Number of {type2plot} Samples per Day (N={n_subjects} subjects)",
    plot_legend=True,
    cal_kwargs=dict(
        color_for_none="#d3d3d3",
        legend_labels_kws=dict(size=6),
    ),
)
plt.savefig(f"../tmp/dayplot_{type2plot}_all.png", dpi=300, bbox_inches='tight')

In [ ]:
df["customer"].nunique()

In [ ]:
type2plot = "Steps"
df_filtered = df[df["type"] == type2plot]
n_subjects = df_filtered["customer"].nunique()
df_dayplot = df_filtered.groupby("startTimestamp_day").size().reset_index(name="n_samples_per_day")


In [ ]:
plot_calendar_by_year(
    df_dayplot,
    cmap="Blues",
    plot_legend=True,
    title=f"Number of {type2plot} Samples per Day (N={n_subjects} subjects)",
    cal_kwargs=dict(color_for_none="#d3d3d3", legend_labels_kws=dict(size=6)),
)
plt.savefig(f"../tmp/dayplot_{type2plot}_all.png", dpi=300, bbox_inches='tight')

plot single subject now

In [ ]:
np.random.seed(45)

customer = df["customer"].iloc[np.random.choice(df.shape[0], 1)[0]]
df_customer = df[df["customer"] == customer]


In [ ]:
type2plot = "HeartRate"
df_filtered = df_customer[df_customer["type"] == type2plot]
df_dayplot = df_filtered.groupby("startTimestamp_day").size().reset_index(name="n_samples_per_day")

plot_calendar_by_year(
    df_dayplot, 
    cmap="Reds", 
    title=f"Number of {type2plot} Samples per Day for Subject {customer}",
    plot_legend=True,
    cal_kwargs=dict(
        color_for_none="#d3d3d3",
        legend_labels_kws=dict(size=6),
    ),
)
plt.savefig(f"../tmp/dayplot_{type2plot}_subject_{customer}.png", dpi=300, bbox_inches='tight')

In [ ]:
type2plot = "Steps"
df_filtered = df_customer[df_customer["type"] == type2plot]
df_dayplot = df_filtered.groupby("startTimestamp_day").size().reset_index(name="n_samples_per_day")

plot_calendar_by_year(
    df_dayplot, 
    cmap="Blues", 
    title=f"Number of {type2plot} Samples per Day for Subject {customer}",
    plot_legend=True,
    cal_kwargs=dict(color_for_none="#e8e8e8")
    # cal_kwargs={"vmax": 5000, "vmin": 1}
)
plt.savefig(f"../tmp/dayplot_{type2plot}_subject_{customer}.png", dpi=300, bbox_inches='tight')

# android vs iphone

In [ ]:
df_redcap_zert = pd.read_csv(datapath + '/redcap/ZERTIFIZIERUNGFOR518_DATA_2025-01-07_1518.csv')
df_redcap = pd.read_csv(datapath + '/redcap/FOR5187_DATA_2025-01-07_1511.csv')
# df_baseline = pd.read_spss(datapath + '/redcap/baseline_T5_data_incl_ns_freezed_241120.sav') # takes supper long to load

In [ ]:
df_redcap["for_id"].value_counts()

In [ ]:
df_redcap

In [ ]:
set(df_redcap_zert["for_id"]) & set(df_redcap["for_id"]) # no intersection

In [ ]:
df_for_smartphone = (
    df_redcap.groupby("for_id")["ema_smartphone"].value_counts().reset_index()
)
df_for_smartphone = pd.concat(
    [
        df_for_smartphone,
        df_redcap_zert.groupby("for_id")["ema_smartphone"].value_counts().reset_index(),
    ],
    ignore_index=True,
)
# only one entry per for_id
assert df_for_smartphone["count"].max() == 1

df_for_smartphone = df_for_smartphone.drop(columns=["count"])

In [ ]:
ema_smartphone_map = {-1: "unknown", 0: "Android", 1: "iPhone"}

In [ ]:
df_for_smartphone

In [ ]:
for_smartphone_unknown = (set(df_redcap["for_id"]) | set(df_redcap_zert["for_id"])) - set(df_for_smartphone["for_id"])
pd.DataFrame({"for_id": list(for_smartphone_unknown), "ema_smartphone": -1})

In [ ]:
df_for_smartphone = pd.concat(
    [
        df_for_smartphone,
        pd.DataFrame({"for_id": list(for_smartphone_unknown), "ema_smartphone": -1}),
    ],
    ignore_index=True,
)

In [ ]:
df_for_smartphone = df_for_smartphone.sort_values("for_id").reset_index(drop=True)

In [ ]:
df_for_smartphone

In [ ]:
df = df.merge(df_for_smartphone, on="for_id", how="left")

## finally plot dayplot of android and iphone

In [ ]:
type2plot = "Longitude"
df_filtered_android = df[(df["type"] == type2plot) & (df["ema_smartphone"] == 0)]
n_subjects_android = df_filtered_android["customer"].nunique()
df_dayplot_android = df_filtered_android.groupby("startTimestamp_day").size().reset_index(name="n_samples_per_day")


In [ ]:
plot_calendar_by_year(
    df_dayplot_android,
    cmap="Greens",
    title=f"Number of GPS Samples per Day for Android Users (N={n_subjects_android})",
    plot_legend=True,
    # cal_kwargs={"vmax": 5000, "vmin": 1}
    cal_kwargs=dict(
        legend_labels_kws=dict(size=6),
        # vmax=31_000,
        # vmin=1,
    ),
)
plt.savefig("../tmp/dayplot_GPS_android.png", dpi=300, bbox_inches='tight')

In [ ]:
type2plot = "Longitude"
df_filtered_iphone = df[(df["type"] == type2plot) & (df["ema_smartphone"] == 1)]
n_subjects_iphone = df_filtered_iphone["customer"].nunique()
df_dayplot_iphone = df_filtered_iphone.groupby("startTimestamp_day").size().reset_index(name="n_samples_per_day")


In [ ]:
assert type2plot == "Longitude"
plot_calendar_by_year(
    df_dayplot_iphone,
    cmap="Blues",
    title=f"Number of GPS Samples per Day for iPhone Users (N={n_subjects_iphone})",
    plot_legend=True,
    # cal_kwargs={"vmax": 5000, "vmin": 1},
    cal_kwargs=dict(
        legend_labels_kws=dict(size=6),
        # vmax=31_000,
        # vmin=1,
    ),
)
plt.savefig("../tmp/dayplot_GPS_iphone.png", dpi=300, bbox_inches='tight')

In [ ]:
type2plot = "Longitude"
df_filtered_unknown = df[(df["type"] == type2plot) & (df["ema_smartphone"] == -1)]
n_subjects_unknown = df_filtered_unknown["customer"].nunique()
df_dayplot_unknown = df_filtered_unknown.groupby("startTimestamp_day").size().reset_index(name="n_samples_per_day")


In [ ]:
plot_calendar_by_year(
    df_dayplot_unknown,
    cmap="Grays",
    title=f"Number of GPS Samples per Day for unknown phone Users (N={n_subjects_unknown})",
    plot_legend=True,
    # cal_kwargs={"vmax": 5000, "vmin": 1},
    cal_kwargs=dict(
        legend_labels_kws=dict(size=6),
        # vmax=31_000,
        # vmin=1,
    ),
)

# cdf like plots for each subject

In [ ]:
df.head()

In [ ]:
# fmt: off
df["customer"] = df["customer"].astype("category")
df["for_id"]   = df["for_id"].astype("category")
df["type"]     = df["type"].astype("category")
# fmt: on

In [ ]:
df_hrperday = df[df["type"] == "HeartRate"].groupby(
    ["customer", "startTimestamp_day"], observed=True
).size().reset_index(name="n_samples_per_day")

In [ ]:
max_n_days = df_hrperday.groupby("customer", observed=True).size().max()
print(f"{max_n_days = }")
df_hrperday.groupby("customer", observed=True).size().sort_values(ascending=False)

In [ ]:
df_customer = df_hrperday[df_hrperday["customer"] == "N3CY"]
plot_calendar_by_year(
    df_customer,
    cmap="Reds",
    # cmap="viridis",
    title=f"Number of Heart Rate Samples per Day for Subject {"N3CY"}",
    plot_legend=True,
    cal_kwargs=dict(
        color_for_none="#d3d3d3",
        legend_labels_kws=dict(size=6),
    ),
)

In [ ]:
df_max_customers_sorted = df_hrperday.groupby("customer",observed=True).max().sort_values(by="n_samples_per_day", ascending=False).reset_index()

In [ ]:
df_max_customers_sorted.iloc[2]["n_samples_per_day"]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
for i, customer in enumerate(df_max_customers_sorted["customer"]):
    df_customer = df_hrperday[df_hrperday["customer"] == customer].sort_values(
        "n_samples_per_day"
    )
    # take color from viridis colormap
    # color = plt.cm.turbo(i / len(df_max_customers_sorted))
    color = plt.cm.viridis_r(
        df_max_customers_sorted.iloc[i]["n_samples_per_day"]
        / df_max_customers_sorted["n_samples_per_day"].max()
    )
    # ax.plot(df_customer["n_samples_per_day"].to_numpy(), label=customer, color=color, alpha=0.5)
    ax.plot(
        np.linspace(0, 1, len(df_customer)),
        df_customer["n_samples_per_day"].to_numpy(),
        label=customer,
        color=color,
        alpha=0.5,
    )
ax.set_xlabel("Days (sorted by number of samples)")
ax.set_xlabel("Percentile of Days (sorted by number of samples)")
ax.set_ylabel("Number of Heart Rate Samples per Day")
ax.set_title("Number of Heart Rate Samples per Day for All Subjects")

In [ ]:
df_customer["n_samples_per_day"].to_numpy()